# Install required libraries

In [1]:
!pip install ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.6/324.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 100.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.9 MB/s eta 0:00:00


# RTDETRL-BASE 5-class relabeled dataset trained best.pt model from Google drive

In [2]:
# !gdown --fuzzy "https://drive.google.com/file/d/14i_3u2z00W_rQ2jzdM6MUC2kRiBPG6DE/view?usp=drive_link"

!gdown --fuzzy "https://drive.google.com/file/d/1bPm0Yl1QhOr7L2wjyDoUVn0a-EJoNaH2/view?usp=drive_link"

Downloading...
From (original): https://drive.google.com/uc?id=1bPm0Yl1QhOr7L2wjyDoUVn0a-EJoNaH2
From (redirected): https://drive.google.com/uc?id=1bPm0Yl1QhOr7L2wjyDoUVn0a-EJoNaH2&confirm=t&uuid=15dd1ec3-78be-45ef-a213-789d1bbf3249
To: /kaggle/working/best.pt
100%|██████████████████████████████████████| 66.2M/66.2M [00:03<00:00, 18.0MB/s]


In [3]:
from roboflow import Roboflow
from pathlib import Path

ROBOFLOW_API_KEY = "xyz"                       # replace the roboflow API key with the provided key
WORKSPACE_ID = "hybridtunedmodeltraining"      # "my-roboflow-workspace"
PROJECT_ID   = "finaldatasetrevised"           #"finalrelabeleddataset" 
VERSION_NUM  = 1                               # dataset version number

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_ID).project(PROJECT_ID)
dataset = project.version(VERSION_NUM).download("yolov8")  # creates a local folder

DATASET_DIR = Path(dataset.location)  # path to downloaded dataset folder
print("Downloaded to:", DATASET_DIR)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to FinalDataSetRevised-1 in yolov8:: 100%|██████████| 29786/29786 [00:12<00:00, 2399.31it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Downloaded to: /kaggle/working/FinalDataSetRevised-1


In [9]:
folder_path = "/kaggle/working/best.pt"
result_folder = 'HybridTunedModelForRelabeledImages'
data_yaml = '/kaggle/working/FinalRelabeledDataSet-1/data.yaml' 

# Load RTDETRL model with the best.py model

In [5]:
import json
import os
from ultralytics import RTDETR
model = RTDETR(os.path.join(folder_path))

# Training the RTDETRL model on the re-labeled five class dataset

Once the model is addequately trained, comment the below cell out

In [13]:
model.train(
    data=data_yaml,
    epochs=30,
    imgsz=640,
    name='RTDETRL_Base',
    project=os.path.join(result_folder,"RTDETRL_Base"),
)

In [ ]:
!tar -czf HybridTunedModelForRelabeledImages.tar.gz \
  -C /kaggle/working/runs/detect \
  HybridTunedModelForRelabeledImages

!ls -lh HybridTunedModelForRelabeledImages.tar.gz

# Downloading the original dataset from roboflow

In [8]:
from roboflow import Roboflow
from pathlib import Path

ROBOFLOW_API_KEY = "xyz"                       # replace the roboflow API key with the provided key
WORKSPACE_ID = "hybridtunedmodeltraining"      # e.g. "my-workspace"
PROJECT_ID   = "regional_road"        # e.g. "uav-cars"
VERSION_NUM  = 3                     # your dataset version number

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_ID).project(PROJECT_ID)
dataset = project.version(VERSION_NUM).download("yolov8")  # creates a local folder

DATASET_DIR = Path(dataset.location)  # path to downloaded dataset folder
print("Downloaded to:", DATASET_DIR)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Regional_Road-3 in yolov8:: 100%|██████████| 29780/29780 [00:13<00:00, 2284.87it/s]


Downloaded to: /kaggle/working/Regional_Road-3


# Detecting the new frames from the original dataset completely

In [20]:
from ultralytics import RTDETR
import yaml
from pathlib import Path
import os, shutil

# Load model
model = RTDETR("/kaggle/working/best.pt")

data_yaml = {
    "train": "images",
    "val": "images",
    "nc": len(model.names),
    "names": [model.names[i] for i in sorted(model.names.keys())]
}

(out_root / "data.yaml").write_text(yaml.dump(data_yaml, sort_keys=False))
print((out_root / "data.yaml").read_text())

src_images = Path("/kaggle/working/Regional_Road-3/train/images")

# Output structure for Roboflow
out_root = Path("/kaggle/working/roboflow_yolov8")
img_out  = out_root / "images"
lbl_out  = out_root / "labels"
img_out.mkdir(parents=True, exist_ok=True)
lbl_out.mkdir(parents=True, exist_ok=True)


def link_or_copy(src: Path, dst: Path):
    try:
        if dst.exists():
            return
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

def unique_stem(p: Path) -> str:
    rel = p.relative_to(src_images)
    stem = "__".join(rel.with_suffix("").parts)
    return stem

pred_iter = model.predict(
    source=str(src_images),
    imgsz=640,
    conf=0.50,
    iou=0.40,
    stream=True,
    save=False,
    save_txt=False,
    batch=4,
    workers=0,
    verbose=False
)

for r in pred_iter:
    p = Path(r.path)
    stem = unique_stem(p)

    img_dst = img_out / f"{stem}{p.suffix.lower()}"
    link_or_copy(p, img_dst)

    txt_path = lbl_out / f"{stem}.txt"

    if r.boxes is None or len(r.boxes) == 0:
        txt_path.write_text("")
        continue

    cls = r.boxes.cls.detach().cpu().numpy().astype(int)
    xywhn = r.boxes.xywhn.detach().cpu().numpy()

    lines = [
        f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}"
        for c, (x, y, w, h) in zip(cls, xywhn)
    ]
    txt_path.write_text("\n".join(lines) + "\n")

print("Folder created.")

{0: 'bus', 1: 'car', 2: 'truck', 3: 'two_wheeler', 4: 'van'}
['bus', 'car', 'truck', 'two_wheeler', 'van']
Folder created.


In [ ]:
# Reuse rf from earlier
workspace = rf.workspace("hybridtunedmodeltraining")

# Upload cleaned dataset to an EXISTING project ID (or new one).
# Tip: use a new project id like "uav-cars-cleaned" so you can compare.
CLEAN_PROJECT_ID = "detection_testing_new"

workspace.upload_dataset(
    "/kaggle/working/roboflow_yolov8",      # dataset path
    CLEAN_PROJECT_ID,    # creates or uploads into this project id
    num_workers=10,
    project_license="MIT",         # change if needed
    project_type="object-detection",
    batch_name="dedup-cleanup",
    num_retries=0,
    is_prediction=False
)

print("Upload finished.")

loading Roboflow workspace...
loading Roboflow project...
Uploading to existing project hybridtunedmodeltraining/detection_testing_new
[UPLOADED] /kaggle/working/roboflow_yolov8/images/00133db5581a7cdd15276c36b4388075_jpg.rf.af8b32d570523149012d7cd42bcb6440.jpg (DNK6C5UUomzvHeqervS9) [1.0s] / annotations = OK [0.7s]
[UPLOADED] /kaggle/working/roboflow_yolov8/images/0031f86c01d74d8974598708b53c24c5_jpg.rf.0116e8bc24e1dbb403b279ac2c0a974b.jpg (2EQNK9YNSvA9AledtueM) [1.4s] / annotations = OK [0.6s]
[UPLOADED] /kaggle/working/roboflow_yolov8/images/003269ef55c991ef7d1fc11c47dcb00e_jpg.rf.babcc9b96ffe0cebb847b02d21106cfc.jpg (Ds6mqo3z1czzfyTJwSLc) [1.4s] / annotations = OK [0.6s]
[UPLOADED] /kaggle/working/roboflow_yolov8/images/002dad5d21070b597b8a7059973428f6_jpg.rf.e82156370885df130107082db5996d1e.jpg (ChS5P46vGymhkdzWwnwr) [1.5s] / annotations = OK [0.6s]
[UPLOADED] /kaggle/working/roboflow_yolov8/images/00272fb1a43dbbc2a2370a9eef1a4b7d_jpg.rf.ca35601979e075976460827b5bde59dc.jpg (RT3wm